In [3]:
from dashscope.audio.asr.translation_recognizer import DASHSCOPE_TRANSLATION_KEY
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_agent
import os
from dotenv import load_dotenv
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

load_dotenv(dotenv_path='.env')
API_KEY = os.getenv("DEEPSEEK_API_KEY")

LLM = init_chat_model(
    model= "deepseek:deepseek-v4-pro",
    api_key=API_KEY,
    base_url="https://api.deepseek.com"
)

prompt = ChatPromptTemplate.from_template("请回答用户的问题:{question}")

chain = prompt | LLM

print(chain.invoke({"question": "篮桥杯的赛道有哪些？"}).content)



蓝桥杯（全称“蓝桥杯全国软件和信息技术专业人才大赛”）的赛道设置比较丰富，主要分为**个人赛**和**团队赛**，涵盖软件、电子、设计等多个领域。目前主要的赛道如下：

### 一、个人赛赛道
1. **软件类**  
   - C/C++程序设计（分研究生组、大学A组、大学B组、大学C组）  
   - Java软件开发（分组同上）  
   - Python程序设计（分组同上）  
   - Web应用开发  
   - 软件测试  

2. **电子类**  
   - 嵌入式设计与开发  
   - 单片机设计与开发  
   - 物联网设计与开发  
   - EDA设计与开发  

### 二、团队赛赛道
1. **数字科技创新赛**  
   涵盖人工智能、区块链、金融科技、元宇宙等多个前沿技术方向，通常以项目形式提交作品。  

2. **视觉艺术设计赛**  
   包含平面设计、视频设计、动画设计、UI设计等赛道，侧重创意与视觉表现。  

### 三、专项赛道
- **青少年创意编程组**（面向中小学生，设有Scratch、Python、C++等科目）  

此外，每年赛道设置可能微调，具体以蓝桥杯官网当年发布的竞赛章程为准。如果你准备参赛，建议先确定自己感兴趣的技术方向，再选择对应的赛道。


In [12]:
import os
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_community.vectorstores import Chroma

# ===================== 配置区 =====================
# 去阿里云 DashScope 控制台拿 API Key，环境变量名必须是这个
os.environ["DASHSCOPE_API_KEY"] = os.getenv("QWEN_API_KEY")
# 千问官方嵌入模型，推荐用 v2 版本，效果更好
EMBED_MODEL = "text-embedding-v2"
# ==================================================

# 1. 模拟私有知识库
knowledge = [
    "蓝桥杯是国内知名算法竞赛，分为Python、C++、Java、单片机等多个组别。",
    "2025年蓝桥杯新增了AI大模型应用赛项，采用线上答题形式。",
    "LangChain是大模型应用开发框架，可用于构建RAG和智能体Agent。",
    "健身增肌需要每日蛋白质摄入达到每公斤体重1.6-2.2克。"
]

# 2. 初始化千问嵌入模型（替换原来的 OpenAIEmbeddings）
embeddings = DashScopeEmbeddings(model=EMBED_MODEL)

# 3. 构建 Chroma 向量库（用法和之前完全一致，只换了嵌入模型）
db = Chroma.from_texts(knowledge, embeddings)

# 4. 封装检索器：输入问题，返回最相关的文本片段
retriever = db.as_retriever(search_kwargs={"k": 2})

# 5. 测试检索
if __name__ == "__main__":
    query = "2025年蓝桥杯有什么新变化？"
    docs = retriever.invoke(query)

    print("你的问题：", query)
    print("\n检索到的相关片段：")
    for i, doc in enumerate(docs):
        print(f"[{i+1}] {doc.page_content}")

你的问题： 2025年蓝桥杯有什么新变化？

检索到的相关片段：
[1] 2025年蓝桥杯新增了AI大模型应用赛项，采用线上答题形式。
[2] 蓝桥杯是国内知名算法竞赛，分为Python、C++、Java、单片机等多个组别。


In [14]:
import os
# 嵌入模型 + 大模型（千问，旧知识复用）
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_community.chat_models import ChatTongyi
# 向量库（旧知识复用）
from langchain_community.vectorstores import Chroma
# 【新知识1】文档加载器：读取本地文件
from langchain_community.document_loaders import TextLoader, PyPDFLoader
# 【新知识2】文本分割器：切分长文档
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 提示词模板（旧知识复用）
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings


# ===================== 配置区 =====================
os.environ["DASHSCOPE_API_KEY"] = "REDACTED"
EMBED_MODEL = "text-embedding-v2"   # 千问嵌入模型
LLM_MODEL = "qwen-turbo"            # 千问大模型，turbo版本快且便宜
FILE_PATH = "/Users/zsh/Desktop/Lumous/PYtest/日报/员工手册202506.pdf"  # 你的手册文件路径
CHUNK_SIZE = 600    # 每个文本块字符数
CHUNK_OVERLAP = 50 # 块之间重叠字符数
TOP_K = 2           # 检索返回最相关的2块
# ==================================================

# ========== 【新知识1】文档加载：从文件读内容 ==========
# 作用：把磁盘上的txt文件，读取成LangChain标准的Document对象
# 完全等价于你之前手动写的 knowledge 列表，只是自动从文件读
loader = PyPDFLoader(FILE_PATH)
raw_docs = loader.load()
print(f"文档加载完成，共 {len(raw_docs)} 份原始文档")

# ========== 【新知识2】文本分割：长文档切成语义小块 ==========
# 为什么要切？一整本手册太长，直接向量化会语义混杂，检索不准
# 递归分割器：优先按段落、换行、句号切割，尽量保证语义完整
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", "。", "，", " ", ""]  # 分割优先级
)
split_docs = text_splitter.split_documents(raw_docs)
print(f"文档分割完成，共 {len(split_docs)} 个文本块")

# ========== 【旧知识复用】构建向量库 + 检索器 ==========
# 和你之前写的一模一样，唯一区别：from_texts → from_documents
embeddings = DashScopeEmbeddings(model=EMBED_MODEL)
db = Chroma.from_documents(split_docs, embeddings)
retriever = db.as_retriever(search_kwargs={"k": TOP_K})

# ========== 【旧知识复用】初始化大模型 + 提示词 ==========
llm = ChatTongyi(model_name=LLM_MODEL, temperature=0)

prompt = ChatPromptTemplate.from_template("""
你是公司行政助手，请严格参考员工手册内容回答用户问题。
规则：
1. 只使用手册里的信息回答，禁止编造
2. 如果手册里没有相关内容，直接回复「暂无相关规定，请咨询人力资源部」
3. 回答条理清晰，简洁明了

员工手册内容：
{context}

用户问题：{question}
""")

# ========== 【旧知识复用】完整RAG问答函数 ==========
def ask_handbook(question):
    # 1. 检索和问题相关的手册片段
    docs = retriever.invoke(question)
    context = "\n---\n".join([doc.page_content for doc in docs])

    # 2. 带着资料调用大模型生成答案
    chain = prompt | llm
    result = chain.invoke({"context": context, "question": question})
    return result.content

# ========== 测试运行 ==========
if __name__ == "__main__":
    question = "员工请假要走什么审批流程？"
    answer = ask_handbook(question)

    print(f"\n问题：{question}")
    print(f"回答：{answer}")

✅ 文档加载完成，共 26 份原始文档
✅ 文档分割完成，共 72 个文本块

问题：员工请假要走什么审批流程？
回答：根据员工手册内容，员工请假的审批流程如下：

1. 员工应提前填写《请假申请表》。
2. 特殊情况下不能提前办理休假手续的，应提前电话通知直接主管和人事。
3. 回公司后应及时补办休假手续。

如手册中未明确规定具体审批层级，可理解为需经直接主管批准。


In [17]:
import os
# ========== 以下组件用法完全不变，你已经掌握 ==========
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_community.chat_models import ChatTongyi
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ========== 【v1.3 修正】Agent 官方标准 API ==========
from langchain.agents import create_agent
from langchain_core.tools import Tool
from langchain_core.messages import HumanMessage

# ===================== 配置区 =====================
os.environ["DASHSCOPE_API_KEY"] = os.getenv("QWEN_API_KEY")
EMBED_MODEL = "text-embedding-v2"
LLM_MODEL = "qwen-turbo"

PDF_PATH = FILE_PATH
PERSIST_DIR = "./handbook_vector_db"  # 向量库持久化目录

CHUNK_SIZE = 600
CHUNK_OVERLAP = 100
TOP_K = 2
# ==================================================


# ========== 1. 构建并持久化向量库（首次运行） ==========
def build_vector_db():
    print("首次运行，正在构建向量库...")
    loader = PyPDFLoader(PDF_PATH)
    raw_docs = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", "。", "，", " ", ""]
    )
    split_docs = text_splitter.split_documents(raw_docs)

    embeddings = DashScopeEmbeddings(model=EMBED_MODEL)
    db = Chroma.from_documents(
        documents=split_docs,
        embedding=embeddings,
        persist_directory=PERSIST_DIR
    )
    print(f"向量库构建完成，已保存到 {PERSIST_DIR}")
    return db


# ========== 2. 加载已有向量库（日常启动用） ==========
def load_vector_db():
    embeddings = DashScopeEmbeddings(model=EMBED_MODEL)
    db = Chroma(
        persist_directory=PERSIST_DIR,
        embedding_function=embeddings
    )
    print("已从本地加载向量库")
    return db


# ========== 3. 封装知识库检索工具（用法完全不变） ==========
def create_handbook_tool(retriever):
    def query_handbook(query: str) -> str:
        """查询员工手册，返回相关原文片段"""
        docs = retriever.invoke(query)
        result = "\n\n".join([
            f"[第{doc.metadata['page']+1}页] {doc.page_content}"
            for doc in docs
        ])
        return result

    handbook_tool = Tool(
        name="employee_handbook",
        description="查询公司员工手册中的规章制度，包括请假、考勤、报销、入职离职等问题。",
        func=query_handbook
    )
    return [handbook_tool]


# ========== 4. 【v1.3 核心修正】创建智能体 ==========
def build_agent(llm, tools):
    system_prompt = """
你是公司行政助手，负责解答员工日常行政问题。
如果问题涉及公司规章制度、请假流程、报销、入职离职等，请调用 employee_handbook 工具查询手册后再回答。
如果是普通闲聊或常识问题，可以直接回答。
回答要简洁清晰，禁止编造手册里没有的内容。
"""

    # LangChain 1.3 官方标准写法：直接 create_agent，无需 AgentExecutor
    agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt=system_prompt
    )
    return agent


# ========== 主程序 ==========
if __name__ == "__main__":
    # 自动判断：有本地向量库就加载，没有就构建
    if not os.path.exists(PERSIST_DIR):
        db = build_vector_db()
    else:
        db = load_vector_db()

    retriever = db.as_retriever(search_kwargs={"k": TOP_K})
    llm = ChatTongyi(model_name=LLM_MODEL, temperature=0)

    # 创建工具 + 构建智能体
    tools = create_handbook_tool(retriever)
    agent = build_agent(llm, tools)

    print("\n===== 智能助手就绪，输入 exit 退出 =====")
    while True:
        user_input = input("\n你：")
        if user_input.lower() == "exit":
            break

        # v1.3 调用方式：传入 messages 列表
        result = agent.invoke({"messages": [HumanMessage(content=user_input)]})
        # 最后一条消息就是最终回答
        answer = result["messages"][-1].content
        print(f"\n助手：{answer}")

已从本地加载向量库

===== 智能助手就绪，输入 exit 退出 =====

助手：你好，我是公司行政助手，负责解答员工日常行政问题。我可以帮助你了解公司规章制度、请假流程、报销、入职离职等相关问题。如果你有其他问题，也可以问我。

助手：根据公司员工手册，关于员工离职的规定如下：

1. **员工个人原因辞职**：
   - 在试用期期间，应提前 **3个工作日** 通知公司；
   - 非试用期期间，应提前 **30天** 以书面形式通知公司。

2. **离职前需完成的事项**：
   - 员工离职前应将 **未休假期休完**。
   - 若与公司签订了《培训协议》且违反服务期约定，应在离职前按照《培训协议》约定支付 **违约金**。

3. **离职交接**：
   - 员工必须办理 **工作和公司资产交接手续** 后方可离职。
   - 需归还所有占有或控制的文件、记录、设备或其他财产，包括但不限于：
     - 公司提供的与工作有关的设备、财产和工具；
     - 与公司产品、技术、财务、人事、运营等相关的所有信息、资料、证件、文件及复印件。

如需进一步了解其他情形（如无过失解除等），可参考员工手册第10.5至10.8条。


KeyboardInterrupt: Interrupted by user